# 01 — EDA & Text Preprocessing
**Proyek:** Analisis Sentimen & Emosi Ulasan E-Commerce Bahasa Indonesia  
**Dataset:** PRDECT-ID Dataset  
**Tim:** Crazy Rich Team — PBA 2026  

---
| Commit | Tahap | File | Isi |
|--------|-------|------|-----|
| **1** | EDA Notebook | `notebooks/01_eda_preprocessing.ipynb` | Load data mentah, plot distribusi kelas, WordCloud, n-gram |
| **2** | Modul Preprocessing | `src/preprocessing.py` | *(dibuat terpisah — lihat file `src/preprocessing.py`)* |
| **3** | Eksekusi & Export | `notebooks/01_eda_preprocessing.ipynb` | Import modul, terapkan `clean_text()`, ekspor CSV |

> **📐 Arsitektur Modular:** Fungsi `clean_text()` dan `batch_clean()` **TIDAK** ditulis di sini,  
> melainkan berada di `src/preprocessing.py`. Notebook ini hanya **mengimpor dan mengeksekusi** modul tersebut.

---
## ⚙️ Setup Environment

In [ ]:
import os
import sys
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ── Tambahkan root proyek ke sys.path ──────────────────────────────────────────
# Wajib dilakukan agar 'from src.preprocessing import ...' bisa berjalan
# dari dalam subdirektori notebooks/
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import emoji
import nltk
from wordcloud import WordCloud
from IPython.display import display

# NLTK data
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# ── Gaya plot ─────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Palet warna per label ─────────────────────────────────────────────────────
SENTIMENT_PALETTE = {'Positive': '#4CAF50', 'Negative': '#F44336'}
EMOTION_PALETTE   = {
    'Happy'  : '#FFD700',
    'Sadness': '#5C85D6',
    'Fear'   : '#9B59B6',
    'Love'   : '#FF69B4',
    'Anger'  : '#E74C3C',
}
SENT_CMAP = {'Positive': 'Greens', 'Negative': 'Reds'}

# ── Path konstanta ────────────────────────────────────────────────────────────
BASE_DIR   = Path('..')
RAW_PATH   = BASE_DIR / 'PRDECT-ID Dataset.csv'
CLEAN_DIR  = BASE_DIR / 'data' / 'clean'
CLEAN_PATH = CLEAN_DIR / 'cleaned_dataset.csv'
FIG_DIR    = BASE_DIR / 'data' / 'figures'

for _d in [CLEAN_DIR, FIG_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print('✅ Setup selesai.')
print(f'   ROOT      : {ROOT}')
print(f'   Dataset   : {RAW_PATH.resolve()}')
print(f'   Clean CSV : {CLEAN_PATH.resolve()}')

---
# 📦 COMMIT 1 — EDA Notebook
> **Git message:** `feat(eda): add exploratory data analysis — distribution, wordcloud, ngram`
```bash
git add notebooks/01_eda_preprocessing.ipynb data/figures/
git commit -m "feat(eda): add exploratory data analysis — distribution, wordcloud, ngram"
```

## 1.1 Load Data Mentah

In [ ]:
df_raw = pd.read_csv(RAW_PATH, sep=';', encoding='utf-8')

n_rows, n_cols = df_raw.shape
print(f'✅ Dataset berhasil dimuat.')
print(f'   Baris  : {n_rows:,}')
print(f'   Kolom  : {n_cols}')
print(f'   Nama   : {df_raw.columns.tolist()}')
df_raw.head(5)

## 1.2 Schema, Kualitas Data & Working Copy

In [ ]:
# ── Schema ringkasan ──────────────────────────────────────────────────────────
schema = pd.DataFrame({
    'dtype'   : df_raw.dtypes,
    'non_null': df_raw.notna().sum(),
    'null'    : df_raw.isna().sum(),
    'null_%'  : (df_raw.isna().sum() / len(df_raw) * 100).round(2),
    'unique'  : df_raw.nunique(),
})
print('Schema DataFrame:')
display(schema)

# ── Missing values ────────────────────────────────────────────────────────────
missing = df_raw.isnull().sum()
missing_df = pd.DataFrame({
    'Jumlah': missing,
    'Persentase (%)': (missing / len(df_raw) * 100).round(2)
})
missing_df = missing_df[missing_df['Jumlah'] > 0]

if missing_df.empty:
    print('\n✅ Tidak ada missing values.')
else:
    print(f'\n⚠️  Missing values pada {len(missing_df)} kolom:')
    display(missing_df)
    print('\nBaris yang mengandung missing value:')
    display(df_raw[df_raw.isnull().any(axis=1)])

# ── Duplikat ──────────────────────────────────────────────────────────────────
n_dup     = df_raw.duplicated().sum()
n_dup_rev = df_raw.duplicated(subset=['Customer Review']).sum()
print(f'\nDuplikat full-row            : {n_dup}')
print(f'Duplikat kolom Customer Review: {n_dup_rev}')

# ── Working copy: hapus missing & duplikat ────────────────────────────────────
df = df_raw.copy()
df.dropna(subset=['Sentiment', 'Emotion', 'Customer Review'], inplace=True)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

removed = len(df_raw) - len(df)
print(f'\nShape setelah cleaning awal : {df.shape}')
print(f'Baris dihapus               : {removed}')

## 1.3 Distribusi Label Target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribusi Label Target — PRDECT-ID', fontsize=14, fontweight='bold')

# ── Sentimen ──────────────────────────────────────────────────────────────────
sent_counts = df['Sentiment'].value_counts()
colors_s    = [SENTIMENT_PALETTE[k] for k in sent_counts.index]
bars = axes[0].bar(sent_counts.index, sent_counts.values,
                   color=colors_s, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, sent_counts.values):
    pct = val / sent_counts.sum() * 100
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 20,
        f'{val:,} ({pct:.1f}%)',
        ha='center', va='bottom', fontsize=10
    )
axes[0].set_title('Sentimen', fontsize=12)
axes[0].set_ylabel('Jumlah Review')
axes[0].set_ylim(0, sent_counts.max() * 1.2)

# ── Emosi ─────────────────────────────────────────────────────────────────────
emo_counts = df['Emotion'].value_counts()
colors_e   = [EMOTION_PALETTE.get(k, '#95A5A6') for k in emo_counts.index]
bars2 = axes[1].bar(emo_counts.index, emo_counts.values,
                    color=colors_e, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars2, emo_counts.values):
    pct = val / emo_counts.sum() * 100
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 10,
        f'{val:,} ({pct:.1f}%)',
        ha='center', va='bottom', fontsize=9
    )
axes[1].set_title('Emosi', fontsize=12)
axes[1].set_ylabel('Jumlah Review')
axes[1].set_ylim(0, emo_counts.max() * 1.25)

plt.tight_layout()
plt.savefig(FIG_DIR / '01_label_distribution.png', bbox_inches='tight')
plt.show()
print('💾 Saved: 01_label_distribution.png')

In [ ]:
# ── Crosstab Emosi × Sentimen ─────────────────────────────────────────────────
ct     = pd.crosstab(df['Emotion'], df['Sentiment'])
ct_pct = ct.div(ct.sum(axis=1), axis=0).mul(100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Crosstab: Emosi x Sentimen', fontsize=13, fontweight='bold')

sns.heatmap(ct,     annot=True, fmt='d',    cmap='YlOrRd',
            ax=axes[0], linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[0].set_title('Jumlah Absolut')

sns.heatmap(ct_pct, annot=True, fmt='.1f', cmap='YlGn',
            ax=axes[1], linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[1].set_title('Proporsi per Baris (%)')

plt.tight_layout()
plt.savefig(FIG_DIR / '02_crosstab_emotion_sentiment.png', bbox_inches='tight')
plt.show()
print('💾 Saved: 02_crosstab_emotion_sentiment.png')

## 1.4 Distribusi Kategori Produk

In [ ]:
cat_counts = df['Category'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 9))
bars = ax.barh(
    cat_counts.index, cat_counts.values,
    color=sns.color_palette('coolwarm', len(cat_counts))
)
for bar, val in zip(bars, cat_counts.values):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        str(val), va='center', fontsize=9
    )
ax.set_xlabel('Jumlah Review')
ax.set_title('Distribusi Kategori Produk', fontweight='bold', fontsize=13)
ax.set_xlim(0, cat_counts.max() * 1.12)
plt.tight_layout()
plt.savefig(FIG_DIR / '03_category_distribution.png', bbox_inches='tight')
plt.show()
print('💾 Saved: 03_category_distribution.png')

## 1.5 Statistik Panjang Teks

In [ ]:
df['char_len'] = df['Customer Review'].str.len()
df['word_len'] = df['Customer Review'].str.split().str.len()

print('Statistik Panjang Review (karakter):')
display(df['char_len'].describe().to_frame('char_len').T.round(1))

print('\nStatistik Panjang Review (kata):')
display(df['word_len'].describe().to_frame('word_len').T.round(1))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Distribusi Panjang Teks per Label', fontsize=13, fontweight='bold')

# ── Histogram karakter & kata per sentimen ────────────────────────────────────
for label, grp in df.groupby('Sentiment'):
    axes[0, 0].hist(grp['char_len'], bins=50, alpha=0.6,
                    label=label, color=SENTIMENT_PALETTE[label])
    axes[0, 1].hist(grp['word_len'], bins=40, alpha=0.6,
                    label=label, color=SENTIMENT_PALETTE[label])

axes[0, 0].set_title('Panjang Karakter per Sentimen')
axes[0, 0].set_xlabel('Jumlah Karakter')
axes[0, 0].legend()
axes[0, 1].set_title('Panjang Kata per Sentimen')
axes[0, 1].set_xlabel('Jumlah Kata')
axes[0, 1].legend()

# ── Boxplot per emosi ─────────────────────────────────────────────────────────
emo_order = df.groupby('Emotion')['char_len'].median().sort_values().index
sns.boxplot(data=df, x='Emotion', y='char_len',
            order=emo_order, palette=EMOTION_PALETTE,
            ax=axes[1, 0], fliersize=2)
axes[1, 0].set_title('Distribusi Karakter per Emosi')
axes[1, 0].set_xlabel('')

sns.boxplot(data=df, x='Emotion', y='word_len',
            order=emo_order, palette=EMOTION_PALETTE,
            ax=axes[1, 1], fliersize=2)
axes[1, 1].set_title('Distribusi Kata per Emosi')
axes[1, 1].set_xlabel('')

plt.tight_layout()
plt.savefig(FIG_DIR / '04_text_length_distribution.png', bbox_inches='tight')
plt.show()
print('💾 Saved: 04_text_length_distribution.png')

## 1.6 Analisis N-Gram (Teks Mentah)

In [ ]:
def get_top_ngrams(series, n=1, top_k=20):
    """Hitung frekuensi n-gram dari pd.Series teks."""
    from nltk.util import ngrams as nltk_ngrams
    counter = Counter()
    for text in series.dropna():
        tokens = str(text).lower().split()
        if n == 1:
            counter.update(tokens)
        else:
            counter.update([' '.join(g) for g in nltk_ngrams(tokens, n)])
    return counter.most_common(top_k)

def plot_ngrams(ax, ngram_data, title, color):
    """Plot horizontal bar chart untuk n-gram."""
    labels, values = zip(*ngram_data)
    y_pos = range(len(labels))
    ax.barh(list(y_pos), values, color=color, edgecolor='white')
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Frekuensi')
    for i, v in enumerate(values):
        ax.text(v + 0.3, i, str(v), va='center', fontsize=8)

# ── Top-20 Unigram per Sentimen ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Top-20 Unigram per Sentimen (Teks Mentah)', fontsize=13, fontweight='bold')

for ax, sent, col in zip(axes, ['Positive', 'Negative'], ['#4CAF50', '#F44336']):
    corpus = df[df['Sentiment'] == sent]['Customer Review']
    top    = get_top_ngrams(corpus, n=1, top_k=20)
    plot_ngrams(ax, top, f'Unigram — {sent}', col)

plt.tight_layout()
plt.savefig(FIG_DIR / '05_top_unigram_raw.png', bbox_inches='tight')
plt.show()
print('💾 Saved: 05_top_unigram_raw.png')

In [ ]:
# ── Top-15 Bigram per Sentimen ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Top-15 Bigram per Sentimen (Teks Mentah)', fontsize=13, fontweight='bold')

for ax, sent, col in zip(axes, ['Positive', 'Negative'], ['#81C784', '#E57373']):
    corpus = df[df['Sentiment'] == sent]['Customer Review']
    top    = get_top_ngrams(corpus, n=2, top_k=15)
    plot_ngrams(ax, top, f'Bigram — {sent}', col)

plt.tight_layout()
plt.savefig(FIG_DIR / '06_top_bigram_raw.png', bbox_inches='tight')
plt.show()
print('💾 Saved: 06_top_bigram_raw.png')

## 1.7 Word Cloud (Teks Mentah)

In [ ]:
# ── WordCloud Keseluruhan ─────────────────────────────────────────────────────
all_text = ' '.join(df['Customer Review'].dropna().astype(str))
wc_all   = WordCloud(
    width=1200, height=600, background_color='white',
    max_words=200, colormap='tab20', random_state=RANDOM_SEED
).generate(all_text)

fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(wc_all, interpolation='bilinear')
ax.axis('off')
ax.set_title('WordCloud — Seluruh Dataset (Teks Mentah)', fontsize=14,
             fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(FIG_DIR / '07_wordcloud_all_raw.png', bbox_inches='tight')
plt.show()
print('💾 Saved: 07_wordcloud_all_raw.png')

In [ ]:
# ── WordCloud per Sentimen ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('WordCloud per Sentimen (Teks Mentah)', fontsize=14, fontweight='bold')

for ax, sent in zip(axes, ['Positive', 'Negative']):
    text = ' '.join(
        df[df['Sentiment'] == sent]['Customer Review'].dropna().astype(str)
    )
    wc = WordCloud(
        width=900, height=500, background_color='white',
        max_words=150, colormap=SENT_CMAP[sent], random_state=RANDOM_SEED
    ).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Sentimen: {sent}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / '08_wordcloud_per_sentiment_raw.png', bbox_inches='tight')
plt.show()
print('💾 Saved: 08_wordcloud_per_sentiment_raw.png')

In [ ]:
# ── WordCloud per Emosi ───────────────────────────────────────────────────────
EMO_BG = {
    'Happy'  : '#FFFDE7', 'Sadness': '#E3F2FD',
    'Fear'   : '#F3E5F5', 'Love'   : '#FCE4EC', 'Anger': '#FFEBEE'
}
EMO_CM = {
    'Happy'  : 'YlOrBr',  'Sadness': 'Blues',
    'Fear'   : 'Purples', 'Love'   : 'RdPu',   'Anger': 'Reds'
}

emotions   = df['Emotion'].value_counts().index.tolist()
ncols      = 3
nrows      = (len(emotions) + ncols - 1) // ncols
fig, axes  = plt.subplots(nrows, ncols, figsize=(17, 5 * nrows))
fig.suptitle('WordCloud per Emosi (Teks Mentah)', fontsize=14, fontweight='bold')
axes_flat  = axes.flatten() if nrows > 1 else list(axes)

for idx, emo in enumerate(emotions):
    text = ' '.join(
        df[df['Emotion'] == emo]['Customer Review'].dropna().astype(str)
    )
    wc = WordCloud(
        width=700, height=400,
        background_color=EMO_BG.get(emo, 'white'),
        max_words=100,
        colormap=EMO_CM.get(emo, 'viridis'),
        random_state=RANDOM_SEED
    ).generate(text)
    axes_flat[idx].imshow(wc, interpolation='bilinear')
    axes_flat[idx].axis('off')
    axes_flat[idx].set_title(f'Emosi: {emo}', fontsize=11, fontweight='bold')

for j in range(len(emotions), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / '09_wordcloud_per_emotion_raw.png', bbox_inches='tight')
plt.show()
print('💾 Saved: 09_wordcloud_per_emotion_raw.png')

## 1.8 Analisis Emoji

In [ ]:
def extract_emojis(text):
    """Kembalikan list emoji yang ada dalam string teks."""
    return [ch for ch in str(text) if ch in emoji.EMOJI_DATA]

df['emojis']      = df['Customer Review'].apply(extract_emojis)
df['emoji_count'] = df['emojis'].str.len()

n_emoji  = (df['emoji_count'] > 0).sum()
pct_emoji = n_emoji / len(df) * 100
avg_emoji = df['emoji_count'].mean()

print(f'Review mengandung emoji : {n_emoji:,} ({pct_emoji:.1f}%)')
print(f'Rata-rata emoji/review  : {avg_emoji:.2f}')

all_emojis = [e for lst in df['emojis'] for e in lst]
top_emojis = Counter(all_emojis).most_common(15)

if top_emojis:
    emo_chars, emo_counts = zip(*top_emojis)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(range(len(emo_chars)), emo_counts,
           color=sns.color_palette('husl', len(emo_chars)))
    ax.set_xticks(range(len(emo_chars)))
    ax.set_xticklabels(list(emo_chars), fontsize=14)
    ax.set_ylabel('Frekuensi')
    ax.set_title('Top-15 Emoji Paling Sering Muncul', fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIG_DIR / '10_top_emoji.png', bbox_inches='tight')
    plt.show()
    print('💾 Saved: 10_top_emoji.png')
else:
    print('ℹ️  Tidak ditemukan emoji dalam dataset.')

## 1.9 Ringkasan Temuan EDA

In [ ]:
sent_ratio  = df['Sentiment'].value_counts(normalize=True).mul(100).round(1)
emo_ratio   = df['Emotion'].value_counts(normalize=True).mul(100).round(1)
n_cats      = df['Category'].nunique()
med_char    = int(df['char_len'].median())
med_word    = int(df['word_len'].median())
max_char    = int(df['char_len'].max())
n_beremoji  = int((df['emoji_count'] > 0).sum())

print('=' * 60)
print('              RINGKASAN TEMUAN EDA')
print('=' * 60)
print(f'  Total sampel        : {len(df):,}')
print(f'  Total kategori      : {n_cats}')
print(f'  Median panjang teks : {med_char} karakter / {med_word} kata')
print(f'  Max panjang teks    : {max_char} karakter')
print(f'  Review beremoji     : {n_beremoji:,}')
print()
print('  Distribusi Sentimen:')
for k, v in sent_ratio.items():
    bar = '#' * int(v / 2)
    print(f'    {k:<12} {v:5.1f}%  {bar}')
print()
print('  Distribusi Emosi:')
for k, v in emo_ratio.items():
    bar = '#' * int(v / 2)
    print(f'    {k:<12} {v:5.1f}%  {bar}')
print('=' * 60)
print()
print('  ⚠️  CATATAN PREPROCESSING (untuk Commit 3):')
print('  1. Dataset sedikit imbalanced (Negative > Positive)')
print('  2. Emosi sangat imbalanced    (Happy >> Anger)')
print('  3. Banyak slang + campur kode Inggris (seller, fast)')
print('  4. Karakter repetisi ditemukan (bagussss, manteppp)')
print('  5. Singkatan dominan: bgs, bgt, yg, gak, udah, dll')

---
# 🧹 COMMIT 3 — Eksekusi Preprocessing & Export
> **Git message:** `feat(preprocessing): apply TextPreprocessor module and export cleaned_dataset.csv`
```bash
git add notebooks/01_eda_preprocessing.ipynb data/clean/cleaned_dataset.csv
git commit -m "feat(preprocessing): apply TextPreprocessor module and export cleaned_dataset.csv"
```

> ⚠️ **Prasyarat:** Pastikan `src/preprocessing.py` sudah ada (Commit 2).  
> Jika kernel di-restart, jalankan ulang **cell Setup** di bagian atas terlebih dahulu.

## 3.1 Import Modul `src.preprocessing`

In [ ]:
# ── Pastikan ROOT ada di sys.path (jika kernel di-restart) ───────────────────
from pathlib import Path
import sys
_ROOT = Path('..').resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

# ── Import fungsi dari src/preprocessing.py (Commit 2) ───────────────────────
from src.preprocessing import clean_text, batch_clean, get_stopwords, get_stemmer

n_sw   = len(get_stopwords())
s_type = type(get_stemmer()).__name__

print('✅ Modul src.preprocessing berhasil diimpor.')
print(f'   Fungsi     : clean_text, batch_clean, get_stopwords, get_stemmer')
print(f'   Stopwords  : {n_sw} kata')
print(f'   Stemmer    : {s_type}')

## 3.2 Sanity Check — Uji `clean_text()` pada Contoh Kalimat

In [ ]:
# ── Kalimat uji mencakup semua karakteristik kotor ulasan e-commerce ──────────
TEST_CASES = [
    ('Barang bagussss bgt!! Penjual ramah & respon cepat',
     'Emoji + karakter repetisi + slang bgt'),
    ('KECEWA!! Barang rusak, gak sesuai deskripsi. Harga 50k tapi jelek bgt',
     'Caps + harga 50k + slang gak/bgt'),
    ('mantep paten joss, fast delivery, packing aman. Top deh seller nya!',
     'Slang mantep/joss + campur kode Inggris'),
    ('udah 3x beli di sini, krn harganya mura tp kualitas oke. Makasih yg jual!',
     'Singkatan ganda: krn, tp, yg, beli'),
    ('bgs banget, harga Rp75.000 worth it. Kemasan aman, ga ada yg rusak.',
     'Slang bgs + format harga Rupiah'),
    ('Alhamdulillah barang ori sesuai gambar, ongkir gratis. recommended!',
     'Ulasan positif semi-formal'),
]

SEP = '-' * 75
print('SANITY CHECK — clean_text() dari src/preprocessing.py')
print('=' * 75)

for raw, keterangan in TEST_CASES:
    hasil = clean_text(raw)
    print(f'[{keterangan}]')
    print(f'  IN : {raw}')
    print(f'  OUT: {hasil}')
    print(SEP)

## 3.3 Terapkan `batch_clean()` ke Seluruh DataFrame

In [ ]:
import time

# ── Siapkan DataFrame output ──────────────────────────────────────────────────
df_clean = df[[
    'Category', 'Product Name', 'Location',
    'Price', 'Overall Rating', 'Number Sold',
    'Customer Rating', 'Customer Review',
    'Sentiment', 'Emotion'
]].copy()

# ── Terapkan preprocessing via batch_clean() dari src/preprocessing.py ────────
print('Menerapkan preprocessing ke seluruh dataset...')
print('(Estimasi waktu: 30–90 detik untuk ~5.000 baris)')
print()

start = time.time()
df_clean['clean_review'] = batch_clean(
    df_clean['Customer Review'],
    verbose=True    # tampilkan progress bar (butuh tqdm)
)
elapsed = time.time() - start

print(f'\n✅ Preprocessing selesai dalam {elapsed:.1f} detik.')
print(f'   Total baris diproses : {len(df_clean):,}')

## 3.4 Statistik & Visualisasi Before vs After

In [ ]:
df_clean['clean_char_len'] = df_clean['clean_review'].str.len()
df_clean['clean_word_len'] = df_clean['clean_review'].str.split().str.len()

# ── Dokumen kosong setelah preprocessing ─────────────────────────────────────
n_empty = (df_clean['clean_review'].str.strip() == '').sum()
print(f'Dokumen kosong setelah preprocessing : {n_empty}')
if n_empty > 0:
    print('Contoh dokumen kosong:')
    display(df_clean[
        df_clean['clean_review'].str.strip() == ''
    ][['Customer Review', 'clean_review']].head(5))

# ── Tabel perbandingan statistik ──────────────────────────────────────────────
compare = pd.DataFrame({
    'Sebelum (karakter)': df_clean['Customer Review'].str.len().describe(),
    'Sesudah (karakter)': df_clean['clean_char_len'].describe(),
    'Sebelum (kata)'    : df_clean['Customer Review'].str.split().str.len().describe(),
    'Sesudah (kata)'    : df_clean['clean_word_len'].describe(),
}).round(1)
print('\nPerbandingan statistik sebelum vs sesudah preprocessing:')
display(compare)

In [ ]:
# ── Histogram Before vs After ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribusi Panjang Teks: Sebelum vs Sesudah Preprocessing',
             fontsize=12, fontweight='bold')

axes[0].hist(df_clean['Customer Review'].str.len(), bins=60,
             alpha=0.55, label='Sebelum', color='#5C85D6')
axes[0].hist(df_clean['clean_char_len'], bins=60,
             alpha=0.55, label='Sesudah', color='#E57373')
axes[0].set_title('Panjang Karakter')
axes[0].set_xlabel('Jumlah Karakter')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()

before_word = df_clean['Customer Review'].str.split().str.len()
axes[1].hist(before_word, bins=50, alpha=0.55, label='Sebelum', color='#5C85D6')
axes[1].hist(df_clean['clean_word_len'], bins=50,
             alpha=0.55, label='Sesudah', color='#E57373')
axes[1].set_title('Panjang Kata')
axes[1].set_xlabel('Jumlah Kata')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / '11_before_after_length.png', bbox_inches='tight')
plt.show()
print('💾 Saved: 11_before_after_length.png')

In [ ]:
# ── Contoh 5 baris sebelum → sesudah ─────────────────────────────────────────
pd.set_option('display.max_colwidth', 110)
sample_compare = df_clean[
    ['Customer Review', 'clean_review', 'Sentiment', 'Emotion']
].sample(5, random_state=RANDOM_SEED).reset_index(drop=True)

print('Contoh Before → After Preprocessing:')
display(sample_compare)

## 3.5 Word Cloud Pasca-Preprocessing

In [ ]:
# ── WordCloud per Sentimen (Teks Bersih) ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('WordCloud per Sentimen — Sesudah Preprocessing',
             fontsize=13, fontweight='bold')

for ax, sent in zip(axes, ['Positive', 'Negative']):
    text = ' '.join(
        df_clean[df_clean['Sentiment'] == sent]['clean_review']
        .dropna().astype(str)
    )
    if text.strip():
        wc = WordCloud(
            width=900, height=500, background_color='white',
            max_words=150, colormap=SENT_CMAP[sent],
            random_state=RANDOM_SEED
        ).generate(text)
        ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'{sent} — Sesudah Preprocessing', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / '12_wordcloud_clean_sentiment.png', bbox_inches='tight')
plt.show()
print('💾 Saved: 12_wordcloud_clean_sentiment.png')

## 3.6 Label Encoding

In [ ]:
# ── Mapping label teks → integer untuk keperluan modeling ────────────────────
SENTIMENT_MAP = {'Positive': 1, 'Negative': 0}
EMOTION_MAP   = {'Happy': 0, 'Sadness': 1, 'Fear': 2, 'Love': 3, 'Anger': 4}

df_clean['sentiment_label'] = df_clean['Sentiment'].map(SENTIMENT_MAP)
df_clean['emotion_label']   = df_clean['Emotion'].map(EMOTION_MAP)

# Drop baris dengan label NaN (nilai tak terduga)
df_clean.dropna(subset=['sentiment_label', 'emotion_label'], inplace=True)
df_clean['sentiment_label'] = df_clean['sentiment_label'].astype(int)
df_clean['emotion_label']   = df_clean['emotion_label'].astype(int)

print('Label Encoding Map:')
print(f'  Sentimen : {SENTIMENT_MAP}')
print(f'  Emosi    : {EMOTION_MAP}')
print(f'  Shape final: {df_clean.shape}')
print()

display(
    df_clean[[
        'Customer Review', 'clean_review',
        'Sentiment', 'sentiment_label',
        'Emotion',   'emotion_label'
    ]].head(5)
)

## 3.7 Export ke `data/clean/cleaned_dataset.csv`

In [ ]:
# ── Pilih kolom yang akan disimpan ────────────────────────────────────────────
EXPORT_COLS = [
    'Category', 'Product Name', 'Location',
    'Price', 'Overall Rating', 'Number Sold', 'Customer Rating',
    'Customer Review',     # Teks asli — untuk audit & traceability
    'clean_review',        # Teks bersih — fitur utama model NLP
    'Sentiment', 'sentiment_label',
    'Emotion',   'emotion_label',
]

df_export = df_clean[EXPORT_COLS].reset_index(drop=True)

# ── Simpan sebagai CSV dengan encoding utf-8-sig (BOM) ────────────────────────
# utf-8-sig: aman dibuka di Excel tanpa masalah karakter Indonesia
df_export.to_csv(CLEAN_PATH, index=False, encoding='utf-8-sig')

size_kb = CLEAN_PATH.stat().st_size / 1024
print('=' * 58)
print('         ✅  EXPORT SELESAI')
print('=' * 58)
print(f'  Path     : {CLEAN_PATH.resolve()}')
print(f'  Baris    : {len(df_export):,}')
print(f'  Kolom    : {len(df_export.columns)}')
print(f'  Ukuran   : {size_kb:.1f} KB')
print(f'  Encoding : utf-8-sig')
print('=' * 58)

In [ ]:
# ── Verifikasi: baca ulang CSV yang baru diekspor ─────────────────────────────
df_verify = pd.read_csv(CLEAN_PATH, encoding='utf-8-sig')

print(f'Verifikasi shape   : {df_verify.shape}')
print(f'Verifikasi kolom   : {df_verify.columns.tolist()}')
print(f'Verifikasi missing : {df_verify.isnull().sum().sum()}')
print()
display(df_verify.head(3))

In [ ]:
# ── Ringkasan akhir eksekusi ──────────────────────────────────────────────────
print('=' * 62)
print('    CHECKPOINT 2 — RINGKASAN EKSEKUSI FINAL')
print('=' * 62)
print(f'  Raw dataset           : {len(df_raw):,} baris')
print(f'  Setelah cleaning awal : {len(df):,} baris')
print(f'  Setelah preprocessing : {len(df_clean):,} baris')
print()
print('  File yang dihasilkan:')
print(f'    data/clean/cleaned_dataset.csv   ({size_kb:.0f} KB)')
print()
print('  Figures tersimpan di data/figures/:')
for fig_file in sorted(FIG_DIR.glob('*.png')):
    print(f'    {fig_file.name}')
print()
print('  Modul yang digunakan:')
print('    src/preprocessing.py  — clean_text(), batch_clean()')
print('=' * 62)
print()
print('  PANDUAN COMMIT:')
print()
print('  # Commit 1 — EDA')
print('  git add notebooks/01_eda_preprocessing.ipynb data/figures/')
print('  git commit -m "feat(eda): add distribution, wordcloud, ngram plots"')
print()
print('  # Commit 2 — Modul Preprocessing (dibuat terpisah)')
print('  git add src/preprocessing.py src/__init__.py')
print('  git commit -m "feat(preprocessing): add clean_text and batch_clean module"')
print()
print('  # Commit 3 — Eksekusi & Export')
print('  git add notebooks/01_eda_preprocessing.ipynb data/clean/')
print('  git commit -m "feat(preprocessing): apply module and export cleaned_dataset.csv"')
print('=' * 62)